# 09 — ct_resid Externalization and Split Storage

NB04 notes that `ct_resid` is visible to anyone with database read access. This notebook asks: **what if `ct_resid` is stored or exposed *outside* the primary record?**

Three scenarios are explored:

1. **Decryption dependency chain** — what can an attacker learn at each level of information?
2. **Split storage** — `ct_resid` in a separate vault from `(qxp, qyp, nonce, tweak)`; blast radius when each store is compromised.
3. **Public API safety** — can `ct_resid` + `nonce` appear in a public response without enabling decryption?

**Key result:** Decrypting `ct_resid` requires `aead_key` *and* `prp_key`. The AEAD's Associated Data is `_build_ad(qx, qy, tweak)`, and recovering `(qx, qy)` from the stored `(qxp, qyp)` requires `prp_key`. Without it, Poly1305 tag verification fails even when `aead_key` is known.

<div style="background:#f5faf9;border:1px solid #b8ddd8;border-radius:8px;padding:12px 14px 14px;margin:10px 0 22px;font-family:sans-serif;">
<div style="font-size:11px;color:#5a9e99;margin-bottom:10px;font-style:italic;">Focus: Lock step — split storage and AEAD–PRP decode dependency</div>
<div style="display:flex;align-items:stretch;">
    <div style="background:#dff0ee;color:#3d7a71;clip-path:polygon(0 0,calc(100% - 22px) 0,100% 50%,calc(100% - 22px) 100%,0 100%);padding:10px 18px 10px 18px;margin-left:0;position:relative;z-index:4;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB02</div><div style="font-weight:500;font-size:13px;">① Project</div></div>
    <div style="background:#dff0ee;color:#3d7a71;clip-path:polygon(0 0,calc(100% - 22px) 0,100% 50%,calc(100% - 22px) 100%,0 100%);padding:10px 18px 10px 18px;margin-left:-21px;position:relative;z-index:3;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB03</div><div style="font-weight:500;font-size:13px;">② Snap+Shuffle</div></div>
    <div style="background:#2a9d8f;color:white;clip-path:polygon(0 0,calc(100% - 22px) 0,100% 50%,calc(100% - 22px) 100%,0 100%);padding:10px 18px 10px 18px;margin-left:-21px;position:relative;z-index:2;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB04</div><div style="font-weight:700;font-size:13px;">③ Lock</div></div>
    <div style="background:#dff0ee;color:#3d7a71;padding:10px 18px 10px 18px;margin-left:-21px;position:relative;z-index:1;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB05</div><div style="font-weight:500;font-size:13px;">④ Wobble</div></div>
</div>
</div>

## Learning Objectives

By the end of this notebook you will be able to:

1. **Identify** the fields assigned to the primary record store and those held in the ct_resid vault in the split-storage architecture.
2. **Explain** the AEAD–PRP mutual dependency and why possession of `aead_key` alone is insufficient to decrypt `ct_resid`.
3. **Demonstrate** the four-level unlock sequence and identify the exact level at which decryption becomes possible.
4. **Analyze** the blast-radius table and rank each compromise scenario by the amount of location information it exposes.
5. **Argue** whether exposing `ct_resid` and `nonce` together in a public API response is safe given the current key architecture.

In [1]:
import struct
import secrets
import pandas as pd

from map_encryption import (
    MapEncryption, SchemeParams,
    _AEAD, _build_ad, _prp_decrypt, _derive_keys,
    _project, _unproject,
)

MASTER_KEY = secrets.token_bytes(32)  # in production: load from secrets manager
enc    = MapEncryption(MASTER_KEY, SchemeParams())
keys   = _derive_keys(MASTER_KEY)
BIN    = enc.params.bin_size_m
ROUNDS = enc.params.prp_rounds

# Load actual cholera death locations; use FID 0 for the dependency-chain demo.
deaths = pd.read_csv('data/cholera_deaths.csv')
demo_row = deaths[deaths['FID'] == 0].iloc[0]
DEMO_LAT, DEMO_LON = demo_row['LAT'], demo_row['LON']
TWEAK = MapEncryption.make_tweak(record_id=int(demo_row['FID']), extra=b'nb09-demo')

record   = enc.encode(DEMO_LAT, DEMO_LON, tweak=TWEAK)
true_pos = enc.decode(record)

print('Record fields:', list(record.keys()))
print(f'Death location (FID 0): {DEMO_LAT:.6f}N  {DEMO_LON:.6f}E')
print(f'Deaths at this address: {int(demo_row["DEATHS"])}')
print(f'True location (decoded): {true_pos[0]:.6f}N  {true_pos[1]:.6f}E')
print(f'ct_resid size : {len(record["ct_resid"])} bytes (16 B plaintext + 16 B Poly1305 tag)')


Record fields: ['qxp', 'qyp', 'nonce', 'ct_resid', 'tweak', 'version']
Death location (FID 0): 51.513418N  -0.137930E
Deaths at this address: 3
True location (decoded): 51.513418N  -0.137930E
ct_resid size : 32 bytes (16 B plaintext + 16 B Poly1305 tag)


## Part 1 — The Decryption Dependency Chain

Decrypting `ct_resid` is not just a matter of having `aead_key`. The AEAD encryption in `encode()` binds the ciphertext to the *original, unshuffled* tile indices via Associated Data:

```python
ct_resid = _AEAD(aead_key).encrypt(nonce,
               struct.pack('>dd', rx, ry),
               _build_ad(qx, qy, tweak))   # AD = pack('>iiI', qx, qy, len(tweak)) + tweak
```

To decrypt, the caller must supply the **same AD**, which requires `(qx, qy)`. These are **not stored** — they must be recovered via:

```python
qx, qy = _prp_decrypt(qxp, qyp, prp_key, tweak, bin_size_m, rounds)
```

An attacker with `aead_key` but without `prp_key` cannot reconstruct `(qx, qy)`. Guessing requires searching ≈ 25.7 billion `(qx, qy)` pairs (M² where M = 160,302). Each wrong guess fails Poly1305 tag verification.

In [2]:
nonce = record['nonce']
ct    = record['ct_resid']
qxp   = record['qxp']
qyp   = record['qyp']
tweak = record['tweak']

# Level 0: ct_resid alone — random nonce, empty AD, no key
r0 = _AEAD(secrets.token_bytes(32)).decrypt(secrets.token_bytes(12), ct, b'')
assert r0 is None
print('Level 0  ct_resid only, random nonce+key :  None ✓')

# Level 1: ct_resid + correct nonce — wrong key
r1 = _AEAD(secrets.token_bytes(32)).decrypt(nonce, ct, b'')
assert r1 is None
print('Level 1  + correct nonce, wrong aead_key :  None ✓')

# Level 2: correct nonce + correct aead_key — no prp_key, so AD is wrong
# The attacker sees (qxp, qyp) in the record but cannot invert the PRP.
wrong_ad = _build_ad(0, 0, tweak)  # guessing qx=qy=0
r2 = _AEAD(keys.aead_key).decrypt(nonce, ct, wrong_ad)
assert r2 is None
print('Level 2  + correct aead_key, no prp_key  :  None ✓  (wrong AD → tag mismatch)')
M = 160_302
print(f'         Brute-force search space: M² = {M**2:,} (qx,qy) pairs')

# Level 3: aead_key + prp_key — reconstruct (qx, qy), build correct AD, decrypt
qx, qy     = _prp_decrypt(qxp, qyp, keys.prp_key, tweak, BIN, ROUNDS)
correct_ad = _build_ad(qx, qy, tweak)
r3         = _AEAD(keys.aead_key).decrypt(nonce, ct, correct_ad)
assert r3 is not None
rx, ry     = struct.unpack('>dd', r3)
lat, lon   = _unproject(qx * BIN + rx, qy * BIN + ry)
assert abs(lat - true_pos[0]) < 1e-9 and abs(lon - true_pos[1]) < 1e-9
print(f'Level 3  + prp_key (correct AD)          :  ({lat:.6f}N, {lon:.6f}E) ✓')
print()
print('The PRP and AEAD are mutually dependent: neither key alone is sufficient.')

Level 0  ct_resid only, random nonce+key :  None ✓
Level 1  + correct nonce, wrong aead_key :  None ✓
Level 2  + correct aead_key, no prp_key  :  None ✓  (wrong AD → tag mismatch)
         Brute-force search space: M² = 25,696,731,204 (qx,qy) pairs
Level 3  + prp_key (correct AD)          :  (51.513418N, -0.137930E) ✓

The PRP and AEAD are mutually dependent: neither key alone is sufficient.


## Decryption Dependency Chain

| Level | ct_resid | nonce | aead_key | prp_key | Can recover (rx, ry)? | Can recover (lat, lon)? |
|-------|:--------:|:-----:|:--------:|:-------:|:---------------------:|:-----------------------:|
| 0 | ✓ | ✗ | ✗ | ✗ | NO | NO |
| 1 | ✓ | ✓ | ✗ | ✗ | NO | NO |
| 2 | ✓ | ✓ | ✓ | ✗ | NO — wrong AD | NO |
| 3 | ✓ | ✓ | ✓ | ✓ | YES | YES |

Level 2 is the key insight: even with `aead_key`, decryption fails because the Associated Data requires `(qx, qy)`, which can only be recovered from `(qxp, qyp)` using `prp_key`.

## Part 2 — Split Storage Architecture

The mutual dependency between PRP and AEAD makes a **split storage** design viable:

| Store | Fields | Tier that needs it |
|-------|--------|--------------------|
| Primary record DB | `qxp`, `qyp`, `nonce`, `tweak`, `version` | Display tier (+ `jitter_key`) |
| ct_resid vault | `record_id` → `ct_resid` | Decode tier only |

The display tier's `render_coordinates` path reads only `qxp`, `qyp`, `nonce` and applies `jitter_key` — **it never touches `ct_resid`**. The vault can therefore live in a higher-security enclave (KMS-backed encrypted store, HSM) that the display layer has no network route to.

**Blast radius:**
- Primary store compromised (no keys): attacker has opaque `(qxp, qyp)` — no location without `prp_key`
- ct_resid vault compromised (no keys): attacker has ciphertexts but **no nonces** — cannot attempt decryption
- Both stores compromised (no keys): full records, still needs `aead_key` + `prp_key` to decode

In [3]:
# Encode the first 10 death records from the cholera dataset.
# Primary store holds display fields (qxp, qyp, nonce, tweak, version).
# ct_resid vault holds encrypted residuals keyed by FID.

sample = deaths.head(10)
all_records = [
    enc.encode(
        row['LAT'], row['LON'],
        tweak=MapEncryption.make_tweak(record_id=int(row['FID']), extra=b'nb09-demo')
    )
    for _, row in sample.iterrows()
]

# Split into two stores
primary_store = [
    {k: r[k] for k in ('qxp', 'qyp', 'nonce', 'tweak', 'version')}
    for r in all_records
]
ct_vault = {
    int(sample.iloc[i]['FID']): all_records[i]['ct_resid']
    for i in range(len(all_records))
}

print(f'Primary store fields : {sorted(primary_store[0])}')
print(f'ct_resid vault keys  : FID → ct_resid  ({len(ct_vault)} entries)')
print()

# Decode tier — join both stores and verify all 10 round-trips
errors = 0
for i, (_, row) in enumerate(sample.iterrows()):
    fid = int(row['FID'])
    full_rec = {**primary_store[i], 'ct_resid': ct_vault[fid]}
    result = enc.decode(full_rec)
    assert result is not None, f'FID {fid} failed to decode'
    lat_err = abs(result[0] - row['LAT'])
    lon_err = abs(result[1] - row['LON'])
    assert max(lat_err, lon_err) < 1e-9
    errors = max(errors, max(lat_err, lon_err))
print(f'Decode tier  — all {len(all_records)} records recovered exactly (max error {errors:.2e} deg)  ✓')

# Primary store alone (zeroed ct) → fails
zeroed = {**primary_store[0], 'ct_resid': bytes(32)}
r_pri = enc.decode(zeroed)
assert r_pri is None
print('Primary store only   — None  ✓  (zeroed ct_resid fails tag verification)')

# ct_resid vault alone → cannot attempt (no nonce or tile indices)
print('ct_resid vault only  — cannot attempt  ✓  (nonce and (qxp,qyp) absent)')

# Display tier — uses primary_store only, no ct_resid
disp = enc.render_coordinates(primary_store[0])
print(f'Display tier (FID 0) — ({disp[0]:.4f}N, {disp[1]:.4f}E)  ✓  (jittered tile, no precise location)')


Primary store fields : ['nonce', 'qxp', 'qyp', 'tweak', 'version']
ct_resid vault keys  : FID → ct_resid  (10 entries)

Decode tier  — all 10 records recovered exactly (max error 1.42e-14 deg)  ✓
Primary store only   — None  ✓  (zeroed ct_resid fails tag verification)
ct_resid vault only  — cannot attempt  ✓  (nonce and (qxp,qyp) absent)
Display tier (FID 0) — (65.2331N, 24.1171E)  ✓  (jittered tile, no precise location)


## Split Storage — Blast Radius by Compromise Scenario

| Attacker has | What they obtain | Can render? | Can decode? |
|--------------|-----------------|:-----------:|:-----------:|
| Primary store, no keys | qxp, qyp, nonce, tweak — opaque without `prp_key` | NO | NO |
| ct_resid vault, no keys | ct_resid only — no nonce, cannot attempt decryption | NO | NO |
| Both stores, no keys | Full record — still needs `aead_key` + `prp_key` | NO | NO |
| Both stores + `aead_key` | Full record + `aead_key` — blocked at AD (no `prp_key`) | NO | NO |
| Both stores + all keys | Full record + both keys — complete access | YES | YES |

## Part 3 — Public API Safety

Some architectures include `ct_resid` in a public API response — for example, returning an encrypted record to the client that submitted the location, for local caching or auditing. Is this safe?

**Nonce safety:** The nonce is already stored unencrypted in every record (`encode()` returns it as a public field). Publishing it adds no attack surface — it is a random salt, not a key.

**ct_resid exposure boundary:** Level 2 established that even with `aead_key`, decryption fails without `prp_key`. Publishing `(ct_resid, nonce)` does not reveal the plaintext residual by itself; the residual becomes recoverable if the required keys and associated data are available. The attack boundary is **key custody plus associated-data reconstruction**, not field visibility alone.

In [4]:
# Simulate a public API response containing ct_resid and nonce
public_response = {
    'record_id': 42,
    'nonce':    record['nonce'],     # already a public record field
    'ct_resid': record['ct_resid'],  # now also returned to caller
}

# Attacker A: has public response + aead_key, no prp_key
guessed_ad = _build_ad(0, 0, b'')   # arbitrary (qx,qy) guess
r_a = _AEAD(keys.aead_key).decrypt(
    public_response['nonce'], public_response['ct_resid'], guessed_ad
)
assert r_a is None
print('Attacker A (aead_key, no prp_key):  None ✓  — cannot build correct AD')

# Attacker B: has public response + both keys (worst case)
qx_b, qy_b = _prp_decrypt(
    record['qxp'], record['qyp'], keys.prp_key, record['tweak'], BIN, ROUNDS
)
r_b = _AEAD(keys.aead_key).decrypt(
    public_response['nonce'],
    public_response['ct_resid'],
    _build_ad(qx_b, qy_b, record['tweak']),
)
assert r_b is not None
print('Attacker B (both keys):             succeeds — key custody is the boundary')
print()
print('Publishing ct_resid + nonce is safe when key material is not exposed.')

Attacker A (aead_key, no prp_key):  None ✓  — cannot build correct AD
Attacker B (both keys):             succeeds — key custody is the boundary

Publishing ct_resid + nonce is safe when key material is not exposed.


## Summary

| Finding | Implication |
|---------|-------------|
| AEAD and PRP are mutually dependent | `aead_key` alone cannot decrypt `ct_resid` — `prp_key` is also required to construct the correct AD |
| Split storage reduces surface area | Display tier never touches `ct_resid`; vault lives in a higher-security enclave without affecting display functionality |
| `ct_resid` + `nonce` in a public API is safe | Key custody is the security boundary, not record field visibility |
| Nonce is already public | It is stored unencrypted in every record; publishing it adds no attack surface |

**Production recommendation:** Store `ct_resid` in a separate vault (encrypted column in an isolated database, or KMS-backed secret store) accessible only to the decode tier. The display tier's entire data path — `qxp`, `qyp`, `nonce`, `jitter_key` — never touches the vault, so a display-tier breach exposes only jittered approximate positions.

## References

- **Snow, J.** (1855). *On the Mode of Communication of Cholera* (2nd ed.). Churchill, London. — Source of the 1854 Soho cholera death and pump location dataset used throughout these notebooks.
- **Nir, Y., & Langley, A.** (2018). ChaCha20 and Poly1305 for IETF Protocols. RFC 8439. IETF. — Specification for the AEAD cipher used in this library.
- **Aumasson, J.-P., Neves, S., Wilcox-O'Halon, Z., & Winnerlein, C.** (2013). BLAKE2: simpler, smaller, fast as MD5. *ACNS 2013*, LNCS 7954, 119–135. — BLAKE2s is the PRF used for key derivation, the Feistel round function, and display jitter.
- **Luby, M., & Rackoff, C.** (1988). How to construct pseudorandom permutations from pseudorandom functions. *SIAM Journal on Computing, 17*(2), 373–386. — Proof that ≥ 4 Feistel rounds produce a PRP.

## Glossary

| Term | Definition |
|------|-----------|
| **`ct_resid`** | The 32-byte AEAD ciphertext of the sub-tile residual `(rx, ry)`; 16 bytes encrypted data + 16 bytes Poly1305 tag. |
| **ct_resid vault** | A separate, higher-security store mapping record ID → `ct_resid`; isolated from the primary record store to reduce blast radius. |
| **Primary store** | The record store holding `qxp`, `qyp`, `nonce`, `tweak`, and `version`; sufficient for the display tier but not for exact coordinate recovery. |
| **Split storage** | An architecture that separates `ct_resid` from the other record fields into two stores with different access controls. |
| **Dependency chain** | The sequence of information required to decrypt `ct_resid`: the ciphertext itself, then `nonce`, then `aead_key`, then `prp_key` (to reconstruct the correct Associated Data). |
| **AEAD-PRP mutual dependency** | The property that decrypting `ct_resid` requires both `aead_key` (for ChaCha20-Poly1305) and `prp_key` (to recover `(qx, qy)` for the Associated Data); neither key alone is sufficient. |
| **Associated Data (AD)** | The binding context `(qx, qy, tweak)` authenticated by the AEAD tag; requires `prp_key` to reconstruct from stored `(qxp, qyp)`. |
| **Public API safety** | The property that including `ct_resid` and `nonce` in a public API response does not enable decryption, because the attack boundary is key custody, not field visibility. |
| **Contextual binding** | The guarantee that a `ct_resid` encrypted under one tweak cannot be decrypted under a different tweak; enforced by including the tweak in the Associated Data. |